# Aerial Swarm Critical-Agent Identification — Training Notebook

This notebook trains a **GraphSAGE → edge-aware attention (GAT-style) → context fusion → per-agent GRU** model on agent-level CSV observations.

**Outputs**
1. `is_critical` — binary classification (BCE loss)
2. `criticality_score` — continuous score (MSE loss)
3. future criticality score — predicts the same agent's score `FUTURE_HORIZON` sampled timesteps ahead (MSE loss)

> **Important:** the supplied schema does not define a `was_promoted` label. This notebook therefore implements a future-score prediction head rather than silently inventing a promotion label. If you later create a valid succession label, you can replace this head with BCE.

The split is by **`case_id`**, not by individual rows, to reduce scenario leakage. Normalization statistics are fitted on training cases only. Each `(case_id, step)` is a graph; edges connect agents within `RADIUS` based on their 3D relative positions. Agent hidden states are tracked by `agent_id` through each scenario's timesteps.

## Before running
- Set `CSV_PATH` in the configuration cell.
- Confirm that `agent_id` persists consistently within each `case_id`.
- Confirm whether `criticality_score` is a meaningful target distinct from `is_critical`.
- This is a practical starting point, not a claim that the architecture is state-of-the-art or novel.

## 1. Install dependencies (run once)

In [ ]:
# Uncomment if needed. Choose a PyTorch build compatible with your machine/CUDA separately if required.
# %pip install pandas numpy scikit-learn matplotlib torch

## 2. Configuration

In [1]:
from pathlib import Path
import copy, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, mean_squared_error,
    mean_absolute_error
)

# CSV_PATH = Path("YOUR_DATASET.csv")  # <-- change this
RADIUS = 55.0
FUTURE_HORIZON = 2       # number of recorded rows/timesteps ahead, not seconds
HIDDEN_DIM = 64
CONTEXT_DIM = 16
DROPOUT = 0.15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 50
PATIENCE = 8
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loss weights. Tune on validation performance, not test performance.
LAMBDA_BINARY = 1.0
LAMBDA_SCORE = 0.5
LAMBDA_FUTURE = 0.25

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


## 3. Load and inspect the dataset

In [3]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Set CSV_PATH to your CSV file. Current value: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

Shape: (1639166, 37)


,case_id,num_agents,n_obstacles,obstacle_density,mean_obstacle_size,clutter_index,obstacle_clustering,attrition_rate,step,agent_id,...,min_neighbor_dist,weighted_density,angular_coverage,criticality_score,degree_centrality,closeness_centrality,betweenness_centrality,eigenvector_centrality,is_critical,was_promoted
0,c001_n60_none,60,2,0.078335,19.78957,0.630676,0.68599,none,0,0,...,16.896336,2.925411,0.479167,0.681887,0.745763,0.797297,0.009607,0.062163,True,False
1,c001_n60_none,60,2,0.078335,19.78957,0.630676,0.68599,none,0,1,...,6.841428,4.717549,0.437500,0.875107,0.677966,0.756410,0.004853,0.185046,True,False
2,c001_n60_none,60,2,0.078335,19.78957,0.630676,0.68599,none,0,2,...,14.777803,1.527958,0.291667,0.673377,0.406780,0.627660,0.001694,0.003368,True,False
3,c001_n60_none,60,2,0.078335,19.78957,0.630676,0.68599,none,0,3,...,17.322876,1.250258,0.208333,0.004849,0.525424,0.678161,0.002434,0.037727,False,False
4,c001_n60_none,60,2,0.078335,19.78957,0.630676,0.68599,none,0,4,...,5.232741,3.218840,0.458333,0.803172,0.627119,0.728395,0.006355,0.013304,True,False



Columns:
['case_id', 'num_agents', 'n_obstacles', 'obstacle_density', 'mean_obstacle_size', 'clutter_index', 'obstacle_clustering', 'attrition_rate', 'step', 'agent_id', 'rel_x', 'rel_y', 'rel_z', 'dist_to_centroid', 'rank_dist_to_centroid', 'dist_to_target', 'rank_dist_to_target', 'vx', 'vy', 'vz', 'speed', 'heading', 'flight_path_angle', 'accel', 'heading_rate', 'num_neighbors', 'mean_neighbor_dist', 'min_neighbor_dist', 'weighted_density', 'angular_coverage', 'criticality_score', 'degree_centrality', 'closeness_centrality', 'betweenness_centrality', 'eigenvector_centrality', 'is_critical', 'was_promoted']


## 4. Define features and targets

The default node features follow the provided schema. Missing optional feature columns are automatically excluded, but the identifiers, positions, and primary labels are required.

- Do **not** use `agent_id` as a numeric feature.
- `criticality_score` is used as a target, not an input, to avoid target leakage.
- Centrality features are included by default; later run an ablation without them to test whether the GNN learns beyond handcrafted centrality.
- The graph is built from `rel_x, rel_y, rel_z`. If those are not actual coordinates in a common frame, replace them with the correct position columns.

In [4]:
ID_COLS = ["case_id", "step", "agent_id"]
POSITION_COLS = ["rel_x", "rel_y", "rel_z"]
BINARY_TARGET = "is_critical"
SCORE_TARGET = "criticality_score"

NODE_FEATURE_CANDIDATES = [
    # relative position / geometry
    "rel_x", "rel_y", "rel_z",
    "dist_to_centroid", "rank_dist_to_centroid",
    "dist_to_target", "rank_dist_to_target",
    # kinematics
    "vx", "vy", "vz", "speed", "heading", "flight_path_angle",
    "accel", "heading_rate",
    # local geometry
    "num_neighbors", "mean_neighbor_dist", "min_neighbor_dist",
    "weighted_density", "angular_coverage",
    # graph centrality
    "degree_centrality", "closeness_centrality",
    "betweenness_centrality", "eigenvector_centrality",
]

ENV_CANDIDATES = [
    "obstacle_density", "mean_obstacle_size", "clutter_index",
    "obstacle_clustering",
]

required = ID_COLS + POSITION_COLS + [BINARY_TARGET, SCORE_TARGET]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

NODE_FEATURES = [c for c in NODE_FEATURE_CANDIDATES if c in df.columns]
ENV_FEATURES = [c for c in ENV_CANDIDATES if c in df.columns]

if not NODE_FEATURES:
    raise ValueError("No node features found.")
print("Node features:", NODE_FEATURES)
print("Environment features:", ENV_FEATURES)

# Basic cleanup and stable ordering.
df = df.dropna(subset=ID_COLS + POSITION_COLS + [BINARY_TARGET, SCORE_TARGET]).copy()
df["step"] = pd.to_numeric(df["step"], errors="raise")
df[BINARY_TARGET] = pd.to_numeric(df[BINARY_TARGET], errors="raise").astype(int)
df[SCORE_TARGET] = pd.to_numeric(df[SCORE_TARGET], errors="raise").astype(float)
df = df.sort_values(["case_id", "step", "agent_id"]).reset_index(drop=True)

# Guard against duplicate agent observations in the same snapshot.
dupes = df.duplicated(["case_id", "step", "agent_id"]).sum()
if dupes:
    raise ValueError(f"Found {dupes} duplicate (case_id, step, agent_id) rows.")

print("Cases:", df["case_id"].nunique())
print("Snapshots:", df.groupby(["case_id", "step"]).ngroups)
print("Positive label rate:", df[BINARY_TARGET].mean())

Node features: ['rel_x', 'rel_y', 'rel_z', 'dist_to_centroid', 'rank_dist_to_centroid', 'dist_to_target', 'rank_dist_to_target', 'vx', 'vy', 'vz', 'speed', 'heading', 'flight_path_angle', 'accel', 'heading_rate', 'num_neighbors', 'mean_neighbor_dist', 'min_neighbor_dist', 'weighted_density', 'angular_coverage', 'degree_centrality', 'closeness_centrality', 'betweenness_centrality', 'eigenvector_centrality']
Environment features: ['obstacle_density', 'mean_obstacle_size', 'clutter_index', 'obstacle_clustering']
Cases: 100
Snapshots: 37731
Positive label rate: 0.38937179028847596


## 5. Derive global context, attrition, and future target

In [10]:
# The context is constant within a scenario snapshot, then broadcast to all agents.
# Attrition is derived from the number of observed agents at each step relative to
# the first observed snapshot in that case. This assumes rows represent alive agents.
counts = (
    df.groupby(["case_id", "step"])["agent_id"].nunique()
      .rename("alive_count").reset_index()
      .sort_values(["case_id", "step"])
)
counts["initial_count"] = counts.groupby("case_id")["alive_count"].transform("first")
counts["attrition_fraction"] = (
    1.0 - counts["alive_count"] / counts["initial_count"].clip(lower=1)
)
counts["lost_since_previous"] = (
    counts.groupby("case_id")["alive_count"].shift(1) - counts["alive_count"]
).fillna(0).clip(lower=0)
counts["attrition_rate"] = (
    counts["lost_since_previous"] / counts["initial_count"].clip(lower=1)
)

# Join count-derived features onto every agent row.
df = df.merge(
    counts[["case_id", "step", "attrition_fraction", "attrition_rate"]],
    on=["case_id", "step"], how="left", validate="many_to_one"
)

# Future criticality score: same agent's score FUTURE_HORIZON observed snapshots later.
# This is a forecast target, not a 'promotion' label.
df["_future_score"] = np.nan
for case_id, case in df.groupby("case_id", sort=False):
    steps = sorted(case["step"].unique())
    step_to_idx = {s: i for i, s in enumerate(steps)}
    future_step = {
        s: steps[i + FUTURE_HORIZON]
        for i, s in enumerate(steps)
        if i + FUTURE_HORIZON < len(steps)
    }
    idx = case.set_index(["step", "agent_id"])[SCORE_TARGET]
    for row_idx, row in case.iterrows():
        fs = future_step.get(row["step"])
        if fs is not None and (fs, row["agent_id"]) in idx.index:
            df.loc[row_idx, "_future_score"] = float(idx.loc[(fs, row["agent_id"])])

CONTEXT_FEATURES = ENV_FEATURES + ["attrition_fraction", "attrition_rate"]
print("Context features:", CONTEXT_FEATURES)
print("Future target availability:", df["_future_score"].notna().mean())

hey
Context features: ['obstacle_density', 'mean_obstacle_size', 'clutter_index', 'obstacle_clustering', 'attrition_fraction', 'attrition_rate']
Future target availability: 0.9945289250753127


## 6. Split by scenario and fit scalers on training data only

All timesteps from one `case_id` stay in the same split. This is critical: randomly splitting rows would put near-duplicate snapshots from the same simulation in train and test and inflate performance.

If the dataset has separate scenario families (for example, held-out swarm sizes or obstacle regimes), prefer a deliberate group split that holds those families out for the final robustness test.

In [11]:
case_ids = df["case_id"].drop_duplicates().to_numpy()
if len(case_ids) < 3:
    raise ValueError("Need at least 3 distinct case_id values for train/validation/test splits.")

train_cases, temp_cases = train_test_split(
    case_ids, test_size=0.30, random_state=SEED
)
val_cases, test_cases = train_test_split(
    temp_cases, test_size=0.50, random_state=SEED
)

train_df = df[df.case_id.isin(train_cases)].copy()
val_df = df[df.case_id.isin(val_cases)].copy()
test_df = df[df.case_id.isin(test_cases)].copy()

node_scaler = StandardScaler()
node_scaler.fit(train_df[NODE_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0))
context_scaler = StandardScaler()
context_scaler.fit(train_df[CONTEXT_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0))

def apply_scalers(frame):
    frame = frame.copy()
    xn = frame[NODE_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    xc = frame[CONTEXT_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    frame["_x"] = list(node_scaler.transform(xn).astype(np.float32))
    frame["_c"] = list(context_scaler.transform(xc).astype(np.float32))
    return frame

train_df = apply_scalers(train_df)
val_df = apply_scalers(val_df)
test_df = apply_scalers(test_df)

print(f"Cases: train={len(train_cases)}, val={len(val_cases)}, test={len(test_cases)}")
print(f"Rows:  train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

Cases: train=70, val=15, test=15
Rows:  train=1102479, val=288749, test=247938


## 7. Snapshot graph construction

For each `(case_id, step)`:
- nodes are the alive agents in that snapshot;
- directed message edges are created for every pair within `RADIUS` (self-loops are handled separately by the self branch);
- edge features encode relative position, distance, relative velocity, and relative velocity projected along the line of sight when velocity columns exist.

The graph is reconstructed from coordinates rather than trusting a precomputed neighbor count. This assumes the 55-unit radius is the intended communication/sensing rule.

In [12]:
def make_snapshot(frame):
    frame = frame.sort_values("agent_id").reset_index(drop=True)
    pos = frame[POSITION_COLS].to_numpy(dtype=np.float32)
    n = len(frame)
    pos_t = torch.tensor(pos, dtype=torch.float32)
    dmat = torch.cdist(pos_t, pos_t)
    src, dst = torch.where((dmat <= RADIUS) & (dmat > 0))
    # src is sender j, dst is receiver i; each directed edge is represented.
    edge_index = torch.stack([src, dst], dim=0) if len(src) else torch.empty((2, 0), dtype=torch.long)

    # Edge features: relative position (sender - receiver), distance,
    # relative velocity (sender - receiver), radial relative velocity.
    rel_pos = pos_t[src] - pos_t[dst] if len(src) else torch.empty((0, 3))
    dist = dmat[src, dst].unsqueeze(1) if len(src) else torch.empty((0, 1))
    if all(c in frame.columns for c in ["vx", "vy", "vz"]):
        vel = torch.tensor(frame[["vx", "vy", "vz"]].to_numpy(dtype=np.float32))
        rel_vel = vel[src] - vel[dst] if len(src) else torch.empty((0, 3))
        radial = (rel_pos * rel_vel).sum(dim=1, keepdim=True) / dist.clamp_min(1e-6)
    else:
        rel_vel = torch.zeros((len(src), 3), dtype=torch.float32)
        radial = torch.zeros((len(src), 1), dtype=torch.float32)
    edge_attr = torch.cat([rel_pos, dist, rel_vel, radial], dim=1)

    x = torch.tensor(np.stack(frame["_x"].to_numpy()), dtype=torch.float32)
    c = torch.tensor(np.stack(frame["_c"].to_numpy()), dtype=torch.float32)
    y = torch.tensor(frame[BINARY_TARGET].to_numpy(dtype=np.float32))
    score = torch.tensor(frame[SCORE_TARGET].to_numpy(dtype=np.float32))
    future = torch.tensor(frame["_future_score"].to_numpy(dtype=np.float32))
    agent_ids = frame["agent_id"].tolist()
    return {
        "x": x, "edge_index": edge_index, "edge_attr": edge_attr,
        "context": c[0], "y": y, "score": score, "future": future,
        "agent_ids": agent_ids, "frame": frame,
    }

def make_case_sequences(frame):
    sequences = {}
    for case_id, case in frame.groupby("case_id", sort=False):
        snapshots = []
        for step, snap in case.groupby("step", sort=True):
            snapshots.append((step, make_snapshot(snap)))
        sequences[case_id] = snapshots
    return sequences

train_sequences = make_case_sequences(train_df)
val_sequences = make_case_sequences(val_df)
test_sequences = make_case_sequences(test_df)

print("Train sequence count:", len(train_sequences))
print("Example snapshots in first train case:", len(next(iter(train_sequences.values()))))

Train sequence count: 70
Example snapshots in first train case: 600


## 8. Model: GraphSAGE → edge-aware attention → context fusion → GRU

The attention layer uses a learned score from `(receiver embedding, sender embedding, edge features)` and normalizes scores over each receiver's incoming neighbors. A separate self projection ensures isolated nodes still receive a valid representation.

A per-agent hidden-state dictionary preserves temporal state by `agent_id` within each case. When an agent disappears, its state is removed. When an agent first appears, its state starts at zero.

In [13]:
class MeanGraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.self_lin = nn.Linear(in_dim, hidden_dim)
        self.neigh_lin = nn.Linear(in_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        n = x.size(0)
        src, dst = edge_index
        agg = torch.zeros_like(x)
        deg = torch.zeros((n, 1), device=x.device, dtype=x.dtype)
        if src.numel():
            agg.index_add_(0, dst, x[src])
            deg.index_add_(0, dst, torch.ones((len(dst), 1), device=x.device, dtype=x.dtype))
        agg = agg / deg.clamp_min(1.0)
        h = self.self_lin(x) + self.neigh_lin(agg)
        return self.dropout(F.relu(self.norm(h)))


class EdgeAwareAttention(nn.Module):
    """GAT-style attention with explicit edge attributes; loops are handled by self projection."""
    def __init__(self, hidden_dim, edge_dim=8, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.edge = nn.Linear(edge_dim, hidden_dim, bias=False)
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1, bias=False)
        )
        self.self_lin = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h, edge_index, edge_attr):
        n = h.size(0)
        src, dst = edge_index
        out = self.self_lin(h)
        if src.numel():
            q = self.q(h[dst])
            k = self.k(h[src])
            e = self.edge(edge_attr)
            logits = self.attn(torch.cat([q, k, e], dim=-1)).squeeze(-1)

            # Stable softmax independently for each receiver node.
            messages = torch.zeros_like(h)
            for i in range(n):
                mask = (dst == i)
                if mask.any():
                    alpha = torch.softmax(logits[mask], dim=0)
                    messages[i] = (alpha.unsqueeze(-1) * self.v(h[src[mask]])).sum(dim=0)
            out = out + messages
        return self.dropout(F.elu(self.norm(out)))


class SwarmCriticalityNet(nn.Module):
    def __init__(self, node_dim, context_dim, edge_dim=8, hidden_dim=64,
                 context_hidden=16, dropout=0.1):
        super().__init__()
        self.node_encoder = nn.Sequential(
            nn.Linear(node_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.sage = MeanGraphSAGE(hidden_dim, hidden_dim, dropout)
        self.gat = EdgeAwareAttention(hidden_dim, edge_dim, dropout)
        self.context_encoder = nn.Sequential(
            nn.Linear(context_dim, context_hidden), nn.ReLU()
        )
        # Mean + max global pooling, node embedding, and context embedding.
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3 + context_hidden, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim)
        )
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)
        self.binary_head = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        self.score_head = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self.future_head = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward_snapshot(self, snap, hidden_by_id):
        x = snap["x"].to(DEVICE)
        edge_index = snap["edge_index"].to(DEVICE)
        edge_attr = snap["edge_attr"].to(DEVICE)
        context = snap["context"].to(DEVICE)
        ids = snap["agent_ids"]

        h = self.node_encoder(x)
        h = self.sage(h, edge_index)
        h = self.gat(h, edge_index, edge_attr)

        # Global graph context: mean + max pooling.
        global_mean = h.mean(dim=0, keepdim=True)
        global_max = h.max(dim=0, keepdim=True).values
        global_vec = torch.cat([global_mean, global_max], dim=-1).expand(h.size(0), -1)

        c = self.context_encoder(context.unsqueeze(0)).expand(h.size(0), -1)
        fused = self.fusion(torch.cat([h, global_vec, c], dim=-1))

        previous = []
        for aid in ids:
            previous.append(hidden_by_id.get(aid, torch.zeros(HIDDEN_DIM, device=DEVICE)))
        h_prev = torch.stack(previous, dim=0)
        h_now = self.gru(fused, h_prev)

        # Drop hidden states for agents absent from this snapshot.
        current_ids = set(ids)
        hidden_by_id = {aid: state for aid, state in hidden_by_id.items() if aid in current_ids}
        for k, aid in enumerate(ids):
            hidden_by_id[aid] = h_now[k]

        return {
            "logits": self.binary_head(h_now).squeeze(-1),
            "score_pred": self.score_head(h_now).squeeze(-1),
            "future_pred": self.future_head(h_now).squeeze(-1),
            "hidden_by_id": hidden_by_id,
        }


model = SwarmCriticalityNet(
    node_dim=len(NODE_FEATURES),
    context_dim=len(CONTEXT_FEATURES),
    edge_dim=8,
    hidden_dim=HIDDEN_DIM,
    context_hidden=CONTEXT_DIM,
    dropout=DROPOUT
).to(DEVICE)

print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

SwarmCriticalityNet(
  (node_encoder): Sequential(
    (0): Linear(in_features=24, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
  )
  (sage): MeanGraphSAGE(
    (self_lin): Linear(in_features=64, out_features=64, bias=True)
    (neigh_lin): Linear(in_features=64, out_features=64, bias=True)
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.15, inplace=False)
  )
  (gat): EdgeAwareAttention(
    (q): Linear(in_features=64, out_features=64, bias=False)
    (k): Linear(in_features=64, out_features=64, bias=False)
    (v): Linear(in_features=64, out_features=64, bias=False)
    (edge): Linear(in_features=8, out_features=64, bias=False)
    (attn): Sequential(
      (0): Linear(in_features=192, out_features=64, bias=True)
      (1): LeakyReLU(negative_slope=0.2)
      (2): Linear(in_features=64, out_features=1, bias=False)
    )
    (self_lin): Linear(in_features=64, out_features=64, bias=True)
    (norm): Laye

## 9. Losses and sequence-level training

Losses are computed per snapshot, then averaged across valid snapshots in a case. Future-score loss ignores rows without a known future observation. Cases are trained one at a time to preserve per-agent GRU state and handle variable graph sizes.

This implementation backpropagates through the full sequence of each case. If scenarios are very long, add truncated backpropagation through time.

In [14]:
def snapshot_loss(out, snap):
    y = snap["y"].to(DEVICE)
    score = snap["score"].to(DEVICE)
    future = snap["future"].to(DEVICE)

    loss_bin = F.binary_cross_entropy_with_logits(out["logits"], y)
    loss_score = F.mse_loss(out["score_pred"], score)

    valid_future = torch.isfinite(future)
    if valid_future.any():
        loss_future = F.mse_loss(out["future_pred"][valid_future], future[valid_future])
    else:
        loss_future = torch.zeros((), device=DEVICE)

    total = (LAMBDA_BINARY * loss_bin
             + LAMBDA_SCORE * loss_score
             + LAMBDA_FUTURE * loss_future)
    return total, {
        "binary": float(loss_bin.detach()),
        "score": float(loss_score.detach()),
        "future": float(loss_future.detach()),
    }


def run_case(model, snapshots, optimizer=None, train=False, collect=False):
    hidden_by_id = {}
    losses = []
    collected = []
    model.train(train)

    context_manager = torch.enable_grad() if train else torch.no_grad()
    with context_manager:
        for step, snap in snapshots:
            out = model.forward_snapshot(snap, hidden_by_id)
            hidden_by_id = out["hidden_by_id"]
            loss, loss_parts = snapshot_loss(out, snap)
            losses.append(loss)
            if collect:
                probs = torch.sigmoid(out["logits"]).detach().cpu().numpy()
                score_pred = out["score_pred"].detach().cpu().numpy()
                future_pred = out["future_pred"].detach().cpu().numpy()
                frame = snap["frame"].copy()
                frame["prob_critical"] = probs
                frame["pred_criticality_score"] = score_pred
                frame["pred_future_score"] = future_pred
                collected.append(frame)

        if not losses:
            return 0.0, collected
        total = torch.stack(losses).mean()

        if train:
            optimizer.zero_grad(set_to_none=True)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

    return float(total.detach().cpu()), collected


def run_epoch(model, sequences, optimizer=None, train=False):
    case_losses = []
    keys = list(sequences.keys())
    if train:
        random.shuffle(keys)
    for case_id in keys:
        loss, _ = run_case(
            model, sequences[case_id], optimizer=optimizer, train=train, collect=False
        )
        case_losses.append(loss)
    return float(np.mean(case_losses)) if case_losses else float("nan")

## 10. Train with validation early stopping

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_val = float("inf")
best_state = None
bad_epochs = 0
history = {"train": [], "val": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_sequences, optimizer=optimizer, train=True)
    val_loss = run_epoch(model, val_sequences, train=False)
    history["train"].append(train_loss)
    history["val"].append(val_loss)

    print(f"Epoch {epoch:03d} | train={train_loss:.4f} | val={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)

plt.figure(figsize=(7, 4))
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training curve")
plt.legend()
plt.grid(True)
plt.show()

Epoch 001 | train=0.5738 | val=0.3559
Epoch 002 | train=0.3479 | val=0.1687
Epoch 003 | train=0.1895 | val=0.1166
Epoch 004 | train=0.1253 | val=0.1014
Epoch 005 | train=0.1132 | val=0.1167
Epoch 006 | train=0.1159 | val=0.1782
Epoch 007 | train=0.0927 | val=0.0954
Epoch 008 | train=0.0909 | val=0.1381
Epoch 009 | train=0.0919 | val=0.0893
Epoch 010 | train=0.0683 | val=0.0726
Epoch 011 | train=0.0626 | val=0.0774
Epoch 012 | train=0.0577 | val=0.1001


## 11. Evaluate on the held-out test cases

In [ ]:
def predict_sequences(model, sequences):
    all_frames = []
    model.eval()
    with torch.no_grad():
        for case_id, snapshots in sequences.items():
            _, frames = run_case(model, snapshots, train=False, collect=True)
            all_frames.extend(frames)
    return pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

test_pred = predict_sequences(model, test_sequences)
display(test_pred[[
    "case_id", "step", "agent_id", BINARY_TARGET, "prob_critical",
    SCORE_TARGET, "pred_criticality_score", "_future_score", "pred_future_score"
]].head())

y_true = test_pred[BINARY_TARGET].to_numpy().astype(int)
p = test_pred["prob_critical"].to_numpy()
y_hat = (p >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_true, y_hat),
    "precision": precision_score(y_true, y_hat, zero_division=0),
    "recall": recall_score(y_true, y_hat, zero_division=0),
    "F1": f1_score(y_true, y_hat, zero_division=0),
    "PR-AUC (average precision)": average_precision_score(y_true, p),
}
if len(np.unique(y_true)) == 2:
    metrics["ROC-AUC"] = roc_auc_score(y_true, p)

valid_score = np.isfinite(test_pred[SCORE_TARGET].to_numpy())
metrics["score MAE"] = mean_absolute_error(
    test_pred.loc[valid_score, SCORE_TARGET],
    test_pred.loc[valid_score, "pred_criticality_score"]
)
metrics["score RMSE"] = mean_squared_error(
    test_pred.loc[valid_score, SCORE_TARGET],
    test_pred.loc[valid_score, "pred_criticality_score"]
) ** 0.5

valid_future = np.isfinite(test_pred["_future_score"].to_numpy())
if valid_future.any():
    metrics["future score MAE"] = mean_absolute_error(
        test_pred.loc[valid_future, "_future_score"],
        test_pred.loc[valid_future, "pred_future_score"]
    )
    metrics["future score RMSE"] = mean_squared_error(
        test_pred.loc[valid_future, "_future_score"],
        test_pred.loc[valid_future, "pred_future_score"]
    ) ** 0.5

pd.Series(metrics, name="test metric").to_frame()

## 12. Ranking metrics: Recall@K and NDCG@K

In [ ]:
def ranking_metrics(frame, k=5, label_col=BINARY_TARGET, score_col="prob_critical"):
    recalls, ndcgs, precisions = [], [], []
    for (_, _), group in frame.groupby(["case_id", "step"]):
        if len(group) == 0:
            continue
        kk = min(k, len(group))
        ranked = group.sort_values(score_col, ascending=False)
        rel = ranked[label_col].to_numpy(dtype=float)
        hits = rel[:kk].sum()
        total_relevant = rel.sum()
        precisions.append(hits / kk)
        recalls.append(hits / total_relevant if total_relevant > 0 else np.nan)
        gains = rel[:kk] / np.log2(np.arange(2, kk + 2))
        dcg = gains.sum()
        ideal = np.sort(rel)[::-1][:kk] / np.log2(np.arange(2, kk + 2))
        idcg = ideal.sum()
        ndcgs.append(dcg / idcg if idcg > 0 else np.nan)
    return {
        f"Precision@{k}": float(np.nanmean(precisions)),
        f"Recall@{k}": float(np.nanmean(recalls)),
        f"NDCG@{k}": float(np.nanmean(ndcgs)),
    }

ranking_metrics(test_pred, k=5)

## 13. Save the model and preprocessing objects

Keep the feature lists and scalers with the weights; inference must use the same feature order and training-set normalization.

In [ ]:
ARTIFACT_PATH = Path("swarm_criticality_model.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "node_features": NODE_FEATURES,
    "context_features": CONTEXT_FEATURES,
    "position_features": POSITION_COLS,
    "radius": RADIUS,
    "hidden_dim": HIDDEN_DIM,
    "context_dim": CONTEXT_DIM,
    "node_scaler_mean": node_scaler.mean_,
    "node_scaler_scale": node_scaler.scale_,
    "context_scaler_mean": context_scaler.mean_,
    "context_scaler_scale": context_scaler.scale_,
    "future_horizon": FUTURE_HORIZON,
}, ARTIFACT_PATH)
print("Saved:", ARTIFACT_PATH.resolve())

test_pred.to_csv("swarm_criticality_test_predictions.csv", index=False)
print("Saved test predictions.")

## 14. Recommended experiments / ablations

Run each variant with the same case-level split and report mean ± standard deviation across several seeds if possible.

1. **MLP only:** no graph message passing.
2. **GraphSAGE only:** remove edge-aware attention.
3. **GraphSAGE + edge-aware attention:** proposed spatial backbone.
4. **No context:** remove obstacle/attrition context.
5. **No centralities:** remove the four centrality columns from `NODE_FEATURE_CANDIDATES`.
6. **No GRU:** classify each snapshot independently.
7. **Generalization tests:** hold out entire swarm sizes, obstacle regimes, or attrition levels if there are enough scenarios.

Report PR-AUC, F1, Recall@K, NDCG@K, and score MAE/RMSE. Accuracy alone is insufficient if critical agents are rare.

### Important limitations to check
- If `is_critical` is a fixed asset identity label, the binary head predicts **high-value asset vs decoy**, not necessarily structural network criticality.
- If `criticality_score` is computed using the same graph features, it may encode a heuristic rather than independent ground truth.
- The future-score target is only available when the same agent is observed `FUTURE_HORIZON` snapshots later.
- If all scenarios share a deterministic label-generation rule, random case splits may still overestimate out-of-distribution generalization. Hold out entire conditions for a stronger test.
- The current attention implementation is intentionally simple and single-head; a production experiment can compare it with a well-tested multi-head GATv2 implementation.